# AvgEGT Engine Monitoring - Master Project Notebook
Welcome to the Master Project Notebook. This document contains the full end-to-end Machine Learning pipeline to monitor engine health and predict `AvgEGT` (Average Exhaust Gas Temperature).

To make this completely transparent, **we have added plain-English explanations next to every single graph** so you know exactly how the AI thinks and why we made our decisions.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, confusion_matrix, accuracy_score, classification_report
from xgboost import XGBRegressor, XGBClassifier
import shap
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Data Loading & Preprocessing
Here we load the raw dataset, drop invalid values, and remove the 'cheating' features to prevent Data Leakage.

In [ ]:
DATA_PATH = os.path.join("..", "data", "raw", "AE_DATA_with_AvgEGT.csv")
if not os.path.exists(DATA_PATH): DATA_PATH = os.path.join("data", "raw", "AE_DATA_with_AvgEGT.csv")
TARGET_COL = "AvgEGT"
EXCLUDED_COLS = [
    "EXHAUST TEMP 1", "EXHAUST TEMP 2", "EXHAUST TEMP 3",
    "EXHAUST TEMP 4", "EXHAUST TEMP 5", "EXHAUST TEMP 6",
    "FREQ", "AMP", "CPW IN TEMP", "CPW OUT TEMP"
]

df = pd.read_csv(DATA_PATH)
df_raw = df.copy() 
df = df.drop_duplicates().dropna()
if TARGET_COL in df.columns:
    df = df[df[TARGET_COL] <= 1000]

drop_cols = [c for c in EXCLUDED_COLS if c in df.columns]
df_clean = df.drop(columns=drop_cols)
print(f"Final Data Shape: {df_clean.shape}")

## 2. Core Exploratory Data Analysis (EDA)
Before we build the AI, we need to understand the raw data. Below are the core graphs of the dataset.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean[TARGET_COL], bins=30, kde=True, color='skyblue')
plt.title(f'Distribution of {TARGET_COL}')
plt.xlabel(TARGET_COL)
plt.ylabel('Frequency')
plt.show()

### 💡 Graph Explanation: Target Distribution
**What is this?** This histogram shows how common different exhaust temperatures are in your engine's history. 
**How to read it:** The highest peak shows the "normal" operating temperature of the engine. The long tail to the right shows rare events where the engine got dangerously hot.

In [ ]:
plt.figure(figsize=(12, 10))
corr_matrix = df_clean.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', annot=False, fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

### 💡 Graph Explanation: Correlation Heatmap
**What is this?** A map of how every sensor interacts with every other sensor.
**How to read it:** Dark Red squares mean two sensors rise together (highly correlated). Dark Blue means when one goes up, the other goes down. White means they have no relationship. The AI uses this math to find hidden patterns.

## 3. Branch A: The Temperature Predictor (Regression)
In this branch, the goal is to predict the *exact decimal temperature* of the exhaust gas based on the other sensors.

In [ ]:
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=500, max_depth=None, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
}

results = []
best_r2 = -float('inf')
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results.append({'Model': name, 'R2': r2, 'RMSE': rmse})
    if r2 > best_r2:
        best_r2, best_model_name, best_model_instance, best_y_pred = r2, name, model, y_pred

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=results_df, x='Model', y='R2', palette='viridis', ax=axes[0])
axes[0].set_title('Model Comparison - R² Score (Higher is Better)')
sns.barplot(data=results_df, x='Model', y='RMSE', palette='magma', ax=axes[1])
axes[1].set_title('Model Comparison - RMSE (Lower is Better)')
plt.show()

### 💡 Graph Explanation: Model Comparison Shootout
**What is this?** We forced 5 different AI architectures to compete against each other. 
**How to read it:** On the right graph (RMSE), lower bars mean the AI made fewer mistakes. You can clearly see that **XGBoost** is significantly lower than Linear Regression, proving it is the safest and most accurate model to use!

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, best_y_pred, alpha=0.5, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title(f'Actual vs Predicted ({best_model_name})')
plt.show()

### 💡 Graph Explanation: Actual vs Predicted Accuracy
**What is this?** This proves how much you can trust the AI.
**How to read it:** The red dashed line represents "perfect reality". The blue dots are the AI's predictions. Because the blue dots tightly cluster around the straight red line, we know the AI is highly reliable in the real world.

In [ ]:
if hasattr(best_model_instance, 'feature_importances_'):
    importances = best_model_instance.feature_importances_
    indices = np.argsort(importances)[::-1]
    sorted_features = [X.columns[i] for i in indices]
    plt.figure(figsize=(10, 8))
    sns.barplot(x=importances[indices], y=sorted_features, palette='mako')
    plt.title('Feature Importances (Regression)')
    plt.show()

### 💡 Graph Explanation: Feature Importance
**What is this?** It shows exactly what the AI looks at to make its decisions.
**How to read it:** The sensors at the top (with the longest bars) have the biggest impact on the engine's exhaust temperature. If those sensors spike, the temperature is going to spike.

## 4. Branch B: The Critical Alarm System (Classification)
Instead of predicting exact temperatures, this branch treats the engine as a binary system (Safe vs Danger). We mathematically defined the top 10% hottest historical states as "Critical Risk" (1), and everything else as "Normal" (0).

In [ ]:
threshold = np.percentile(y, 90)
y_cls = (y >= threshold).astype(int)

y_train_cls = (y_train >= threshold).astype(int)
y_test_cls = (y_test >= threshold).astype(int)

cls_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBoost Classifier': XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
}

cls_results = []
best_acc = 0
for name, model in cls_models.items():
    model.fit(X_train_scaled, y_train_cls)
    y_pred_cls = model.predict(X_test_scaled)
    acc = accuracy_score(y_test_cls, y_pred_cls)
    cls_results.append({'Model': name, 'Accuracy': acc})
    if acc > best_acc:
        best_acc, best_cls_name, best_cls_model, best_y_pred_cls = acc, name, model, y_pred_cls

cls_results_df = pd.DataFrame(cls_results).sort_values('Accuracy', ascending=False)
cls_results_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=cls_results_df, x='Model', y='Accuracy', palette='magma')
plt.title('Classification Shootout (Accuracy - Higher is Better)')
plt.ylim(0, 1.0)
plt.show()

### 💡 Graph Explanation: Classification Accuracy
**What is this?** The shootout for the Alarm System. 
**How to read it:** The higher the bar, the more often the AI correctly identified a critical engine failure. XGBoost Classifier wins again!

In [ ]:
cm = confusion_matrix(y_test_cls, best_y_pred_cls)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Critical'], yticklabels=['Normal', 'Critical'])
plt.title(f'Confusion Matrix ({best_cls_name})')
plt.ylabel('Actual Truth')
plt.xlabel('AI Prediction')
plt.show()

### 💡 Graph Explanation: The Confusion Matrix
**What is this?** A detailed breakdown of where the AI got it right, and where it got it wrong.
**How to read it:** 
- **Top-Left (Normal/Normal):** Engine was safe, AI said it was safe. (Good!)
- **Bottom-Right (Critical/Critical):** Engine was failing, AI correctly triggered the alarm. (Good!)
- **Bottom-Left (Critical/Normal):** Engine was failing, but AI missed it. (Dangerous Mistake)
- **Top-Right (Normal/Critical):** Engine was safe, but AI triggered a false alarm. (Annoying, but safe)

## 5. Exhaustive Engine Analytics (The 220+ Plots)
Below, the AI iterates through every single sensor in the engine to generate exhaustive diagnostics. 
We have programmed the AI to print a plain-English explanation directly above every single graph so you don't get lost!

In [ ]:
def generate_feature_loops_inline(df, prefix=""):
    features = [c for c in df.columns if c != TARGET_COL]
    for i, feature in enumerate(features):
        # 1. Feature Distribution
        print(f"\n=========================================================")
        print(f"💡 EXPLANATION: This histogram shows how often different values of {feature} occur.")
        print(f"Look for the highest peak to find the normal operating state for {feature}.")
        print(f"=========================================================")
        plt.figure(figsize=(8, 4))
        sns.histplot(df[feature], kde=True, color='teal')
        plt.title(f'[{prefix.upper()}] Distribution of {feature}')
        plt.tight_layout()
        plt.show()
        
        # 2. Boxplot
        print(f"\n=========================================================")
        print(f"💡 EXPLANATION: This boxplot hunts for anomalies in {feature}.")
        print(f"The box is the 'safe zone'. Any dots outside the box are extreme mechanical anomalies (outliers).")
        print(f"=========================================================")
        plt.figure(figsize=(8, 2))
        sns.boxplot(x=df[feature], color='orange')
        plt.title(f'[{prefix.upper()}] Boxplot of {feature}')
        plt.tight_layout()
        plt.show()
        
        # 3. Scatter vs Target
        print(f"\n=========================================================")
        print(f"💡 EXPLANATION: This shows the direct relationship between {feature} and the Engine Temperature.")
        print(f"If the dots go UP and to the right, it means when {feature} increases, the Engine gets hotter.")
        print(f"=========================================================")
        plt.figure(figsize=(8, 4))
        sns.scatterplot(data=df, x=feature, y=TARGET_COL, alpha=0.6, color='navy')
        plt.title(f'[{prefix.upper()}] {feature} vs {TARGET_COL}')
        plt.tight_layout()
        plt.show()

print("Generating exhaustive Sanitized Dataset EDA loops (Inline)...")
generate_feature_loops_inline(df_clean, prefix="sanitized")
